In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
df=spark.read.format("csv")\
    .option("inferSchema",True)\
    .option("header",True)\
    .load("/Volumes/databrickscatalog/bronze/bronze_volume/customer/")


In [0]:
display(df)
# display is a method of the DataFrame class to display the DataFrame

In [0]:
df.printSchema()
# printSchema is a method of the DataFrame class to print the schema.

In [0]:
df= df.withColumn("name",upper(col("name")))
display(df)

In [0]:
df=df.withColumn("domain",split(col("email"),"@")[1])
display(df)

In [0]:
display(
    df.groupBy("domain")
    .agg(count(col("customer_id")).alias("total_customer"))
    .sort(col("total_customer").desc())
)

Databricks visualization. Run in Databricks to view.

In [0]:
df=df.withColumn("processDate",current_timestamp())
display(df)

In [0]:
if spark.catalog.tableExists("databrickscatalog.sliver.customers_sliver"):
    dlt_obj= DeltaTable.forName(spark,"databrickscatalog.sliver.customers_sliver")
    dlt_obj.alias("tgt").merge(df.alias("src"),"tgt.customer_id=src.customer_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("databrickscatalog.sliver.customers_sliver")

In [0]:
%sql 
select count(*) from databrickscatalog.sliver.customers_sliver;

In [0]:
df_prod=spark.read.format("csv")\
    .options(header="true",inferSchema="true")\
    .load("/Volumes/databrickscatalog/bronze/bronze_volume/products/")

df_prod=df_prod.withColumn("processDate",current_timestamp())

display(df_prod)


In [0]:
df_prod=display(
    df_prod.groupBy("category")
    .agg(avg(col("price")).alias("avg_price"))
)

Databricks visualization. Run in Databricks to view.

In [0]:
if spark.catalog.tableExists("databrickscatalog.sliver.products_sliver"):
    dlt_obj= DeltaTable.forName(spark,"databrickscatalog.sliver.products_sliver")
    dlt_obj.alias("tgt").merge(df_prod.alias("src"),"tgt.product_id=src.product_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_prod.write.format("delta")\
        .mode("append")\
        .saveAsTable("databrickscatalog.sliver.products_sliver")

In [0]:
df_str=spark.read.format("csv")\
    .options(header="true",inferSchema="true")\
    .load("/Volumes/databrickscatalog/bronze/bronze_volume/stores/")

display(df_str)

In [0]:
df_str=df_str.withColumn("store_name",regexp_replace(col("store_name"),"_",""))
display(df_str)

In [0]:
df_str=df_str.withColumn("processDate",current_timestamp())
display(df_str)


In [0]:
if spark.catalog.tableExists("databrickscatalog.sliver.stores_sliver"):
    dlt_obj= DeltaTable.forName(spark,"databrickscatalog.sliver.stores_sliver")
    dlt_obj.alias("tgt").merge(df_str.alias("src"),"tgt.store_id=src.store_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_str.write.format("delta")\
        .mode("append")\
        .saveAsTable("databrickscatalog.sliver.stores_sliver")

In [0]:
df_sal=spark.read.format("csv")\
    .options(header="true",inferSchema="true")\
    .load("/Volumes/databrickscatalog/bronze/bronze_volume/sales/")
# display(df_sal)
df_sal=df_sal.withColumn("pricePerSale",round(col("total_amount")/col("quantity"),2))
df_sal=df_sal.withColumn("processDate",current_timestamp())
display(df_sal)

if spark.catalog.tableExists("databrickscatalog.sliver.sales_sliver"):
    dlt_obj= DeltaTable.forName(spark,"databrickscatalog.sliver.sales_sliver")
    dlt_obj.alias("tgt").merge(df_sal.alias("src"),"tgt.sales_id=src.sales_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_sal.write.format("delta")\
        .mode("append")\
        .saveAsTable("databrickscatalog.sliver.sales_sliver")

In [0]:
display(spark.sql("select * from databrickscatalog.sliver.sales_sliver"))